In [ ]:
import os
import sys
import boto3
from boto3.dynamodb.conditions import Key

In [ ]:
# Initialize project root
project_root = "/home/lpascual/Projects/E-Commerce_DWH/scripts"

if project_root not in sys.path:
    sys.path.append(project_root)

In [ ]:
from infrastructure.schemas.dynamoDB.job_schemas import job_schemas

In [25]:
os.environ.setdefault("AWS_PROFILE", "lpascual")

'lpascual'

In [26]:
dynamo_client = boto3.client("dynamodb")

### Confirm if Jobs table exist
---

In [27]:
dynamo_client.list_tables()

{'TableNames': [],
 'ResponseMetadata': {'RequestId': 'IRLALI3U17D0JPVNNEFPTRRVUFVV4KQNSO5AEMVJF66Q9ASUAAJG',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'server': 'Server',
   'date': 'Fri, 13 Mar 2026 15:51:57 GMT',
   'content-type': 'application/x-amz-json-1.0',
   'content-length': '17',
   'connection': 'keep-alive',
   'x-amzn-requestid': 'IRLALI3U17D0JPVNNEFPTRRVUFVV4KQNSO5AEMVJF66Q9ASUAAJG',
   'x-amz-crc32': '1315925753'},
  'RetryAttempts': 0}}

In [28]:
tables = dynamo_client.list_tables()["TableNames"]

In [29]:
"ecom_jobs" in tables

False

### Create Job table
---

In [30]:
if "ecom_jobs" not in tables:
    response = dynamo_client.create_table(
        AttributeDefinitions=[
            {
                "AttributeName": "PK",
                "AttributeType": "S"
            },
            {
                "AttributeName": "SK",
                "AttributeType": "S"
            }            
        ],
        TableName="ecom_jobs",
        KeySchema=[
            {
                "AttributeName": "PK",
                "KeyType": "HASH"
            },
            {
                "AttributeName": "SK",
                "KeyType": "RANGE"
            }            
        ],
        BillingMode="PAY_PER_REQUEST"
    )
    print(response)

{'TableDescription': {'AttributeDefinitions': [{'AttributeName': 'PK', 'AttributeType': 'S'}, {'AttributeName': 'SK', 'AttributeType': 'S'}], 'TableName': 'ecom_jobs', 'KeySchema': [{'AttributeName': 'PK', 'KeyType': 'HASH'}, {'AttributeName': 'SK', 'KeyType': 'RANGE'}], 'TableStatus': 'CREATING', 'CreationDateTime': datetime.datetime(2026, 3, 13, 8, 51, 57, 254000, tzinfo=tzlocal()), 'ProvisionedThroughput': {'NumberOfDecreasesToday': 0, 'ReadCapacityUnits': 0, 'WriteCapacityUnits': 0}, 'TableSizeBytes': 0, 'ItemCount': 0, 'TableArn': 'arn:aws:dynamodb:us-west-1:060638774782:table/ecom_jobs', 'TableId': '4bcbd873-55c9-4a9a-9cb1-b45da5cc73b8', 'BillingModeSummary': {'BillingMode': 'PAY_PER_REQUEST'}, 'DeletionProtectionEnabled': False}, 'ResponseMetadata': {'RequestId': 'TJ6PI09CBF30IH2MP8RB3AP8PFVV4KQNSO5AEMVJF66Q9ASUAAJG', 'HTTPStatusCode': 200, 'HTTPHeaders': {'server': 'Server', 'date': 'Fri, 13 Mar 2026 15:51:57 GMT', 'content-type': 'application/x-amz-json-1.0', 'content-length':

In [31]:
tables = dynamo_client.list_tables()["TableNames"]
"ecom_jobs" in tables

True

In [32]:
dynamodb_resource = boto3.resource("dynamodb")
ecom_jobs_table = dynamodb_resource.Table("ecom_jobs")

In [43]:
# Insert job meta records into table
for job_db_schema in job_schemas:
    response = ecom_jobs_table.put_item(Item=job_db_schema)
    print(f"{job_db_schema['PK']}: Status Code ({response['ResponseMetadata']['HTTPStatusCode']})")

JOB#INGESTION: Status Code (200)
JOB#BRONZETRANSFORM: Status Code (200)
JOB#SILVERTRANSFORM: Status Code (200)
JOB#GOLDTRANSFORM: Status Code (200)
JOB#COPYTOREDSHIFT: Status Code (200)


In [ ]:
for job_db_schema in job_schemas:
    response = ecom_jobs_table.query(
        KeyConditionExpression=Key("PK").eq(f"{job_db_schema['PK']}") & Key("SK").eq("META")
    )

    items = response.get("Items", [])
    for item in items:
        print(f"{item['PK']} : {item['description']}")

JOB#INGESTION : Converts csv files to parquet format
JOB#BRONZETRANSFORM : Applies Schema Enforcement & Data Quality Checks
JOB#SILVERTRANSFORM : Conforms data into a normalized schema & applies business logic
JOB#GOLDTRANSFORM : Conforms data into a dimensional schema
JOB#COPYTOREDSHIFT : Migrates data into Redshift
